# E08 Bike Sharing：时间序列划分、滞后特征与泄漏检测

> Issue #9 / Epic #6 / 路线图 E08

## 你将学到
1. **禁止随机打乱**：小时级租车 demand 强自相关；必须按时间切分或 `TimeSeriesSplit`。
2. **滞后特征**：`cnt_lag1`、`cnt_lag24` 只能用 **过去** 的 `shift`，滚动均值亦须 `shift(1)` 后再 rolling。
3. **`extract_datetime_features`**：统一从 datetime 派生 hour/weekday/month 等，避免手写散落。
4. **`LeakageDetector.detect_time_leakage`**：检查可疑列名与未来相关结构。
5. **天气/节假日 EDA**：`temp`/`hum`/`workingday`/`holiday` 与 `cnt` 的关系。

## 数据路径
- 本地 `data/bike_sharing_hour.csv`
- OpenML `Bike_Sharing_Demand`
- 回退：**合成 DEMO** 时间序列（与 _deepen_all 一致）

## 理论导读

时间序列 ML 的黄金法则：**验证集时间必须晚于训练集**。随机 `train_test_split(shuffle=True)` 会用「明天」预测「昨天」，离线分数虚高。

滞后特征本质是自回归信号；错误写法 `shift(-1)` 或 `rolling` 未先 shift 会把 **未来 cnt** 泄漏进特征。

常用指标：MAE + **RMSLE**（偏好相对误差，惩罚低估高峰）。

## 常见误区
- 把 `casual` + `registered` 当特征（它们之和即 `cnt`，训练 Kaggle 题时属目标泄漏）。
- 在划分前对全序列做 `cnt` 标准化。

## 深入：小时 demand 的周期结构

日周期：早晚高峰对应 `datetime_hour`（或 `hour`）正弦分量；周周期：`workingday` 与周末 `cnt` 常不同。天气：`temp` 极端时骑行意愿变化——但注意 **因果与混杂**（雨天同时影响 `hum` 与出行）。

**滞后阶数选择**：`lag24` 对齐「昨日同时」；`roll24` 平滑单日噪声。更长滞后（168 小时）可捕周周期，但缩短有效训练样本（dropna 增多）。

**DEMO 合成数据**：无 CSV 时正弦 + 噪声仍可演示 **时间切分 vs 随机切分** 的流程差异；换真实 Kaggle `hour.csv` 后重点检查 `casual`/`registered` 是否误作特征。

**wanglaoshi 时间 API**：`extract_datetime_features` 生成的列名带前缀（如 `datetime_hour`），建模时与其他 `hr` 列勿重复共线。


In [ ]:
QUICK = True
RANDOM_STATE = 42

# 一键就绪：关警告 + 打印环境 + 设种子（勿再叠 JE.no_warning / filterwarnings）
from wanglaoshi.runtime import ready
ready(seed=RANDOM_STATE)


## 课程研讨笔录（E08 专用）

### 课堂问答题
1. 若特征含 `cnt_roll24` 但未 `shift(1)`，离线分异常高——如何用 `detect_time_leakage` 快速筛查？
2. 时间切分 80/20 与 `TimeSeriesSplit` 各适合什么阶段？
3. DEMO 合成序列通过 evidence，说明什么？

### 实验记录模板
- 数据源：local / OpenML / DEMO
- `extract_datetime_features` 列名列表
- 切分：时间有序 80/20，无 shuffle
- 泄漏：安全滞后 vs 故意 `shift(-1)` 列 warnings
- 指标：MAE、RMSLE；`time_series_cv` mean MSE
- 提交 validate

### 特征工程清单
- 允许：weather、日历、shift 后滞后
- 禁止：casual+registered、未 shift 的 rolling cnt
- 可选：hour×workingday 交互

### 与路线图映射
- 时间序列 CV 课
- FE 时间 API 课
- LeakageDetector 课

### 打分 rubric
| 维度 | 达标 | 优秀 |
|------|------|------|
| 时间序 | 排序+切分 | 解释高峰预测 |
| 滞后 | shift 正确 | 泄漏对比实验 |
| API | datetime FE + detect | time_series_cv |
| EDA | 天气/工作日 | holiday 讨论 |

### 运营比喻
把 `cnt` 想成共享单车驻点库存：低估高峰 RMSLE 惩罚大，意味着「缺车」比「多车」更痛——指标选择与商业对齐。

### 复现口令
无 csv 时 DEMO=True 仍可提交；换真实 hour.csv 时第一时间删泄漏列。

（本节补充 ~3× 篇幅中的时间序列理论叙述。）

### 教师讲稿节选

共享单车 demand 是时间序列入门的「甜点」：周期清晰、可解释特征多、泄漏方式典型。第一分钟强调 **时间轴不能 shuffle**；第二分钟画 lag 示意图：t 时刻只能看见 t-1 的 cnt。第三分钟介绍 wanglaoshi 时间特征 API，避免学生在不同案例里手写 `dt.hour` 口径不一。第四分钟 `detect_time_leakage`：故意构造 `shift(-1)` 列，让检测器响 —— 比抽象讲「未来函数」更有肌肉记忆。第五分钟 RMSLE vs MAE：高峰预测低估在 RMSLE 下更痛。DEMO 模式要直言：合成数据只保流程，不保数值可发表；换真数据后第一件事是删 `registered`/`casual` 目标分解列。

### 学周作业
用同一管线对比「仅日历特征」与「日历+滞后」的 MAE；写 200 字说明提升是否来自合法信息。


## 1. 加载数据（本地 → OpenML → DEMO 合成）


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

path = Path('../../data/bike_sharing_hour.csv')
DEMO = False
if path.exists():
    df = pd.read_csv(path)
    print('local', df.shape)
else:
    try:
        from sklearn.datasets import fetch_openml
        raw = fetch_openml('Bike_Sharing_Demand', version=2, as_frame=True, parser='auto')
        df = raw.frame.copy()
        print('OpenML', df.shape)
    except Exception:
        DEMO = True
        print('使用合成时间序列 DEMO')
        n = 2000 if QUICK else 5000
        rng = np.random.RandomState(RANDOM_STATE)
        idx = pd.date_range('2012-01-01', periods=n, freq='h')
        df = pd.DataFrame({
            'datetime': idx,
            'temp': 20 + 10 * np.sin(np.arange(n) / 24) + rng.normal(0, 2, n),
            'hum': 60 + rng.normal(0, 10, n),
            'windspeed': np.abs(rng.normal(10, 3, n)),
            'season': ((idx.month % 12) // 3 + 1),
            'holiday': 0,
            'workingday': (idx.dayofweek < 5).astype(int),
            'cnt': None,
        })
        df['cnt'] = (
            100 + 40 * np.sin(np.arange(n) / 24) + 20 * df['workingday'] + rng.normal(0, 15, n)
        ).clip(0, None)

if 'cnt' not in df.columns:
    for c in ['count', 'Count', 'total']:
        if c in df.columns:
            df = df.rename(columns={c: 'cnt'})
            break
print(df.head())
print('DEMO', DEMO)

import hashlib
from pathlib import Path as _P
data_mode = 'demo' if DEMO else 'real'
n_rows = int(len(df))
data_source = str(path) if 'path' in dir() else 'unknown'
try:
    _fp = path if path.exists() else None
    data_sha256 = hashlib.sha256(_fp.read_bytes()).hexdigest() if _fp else None
except Exception:
    data_sha256 = None
print({'data_mode': data_mode, 'n_rows': n_rows, 'data_source': data_source, 'data_sha256': (data_sha256 or '')[:16]})


## 2. 时间排序与 wanglaoshi 时间特征


In [ ]:
from wanglaoshi import features as FE

work = df.copy()
time_col = None
for c in ['dteday', 'datetime', 'date', 'timestamp']:
    if c in work.columns:
        time_col = c
        break
if time_col:
    work[time_col] = pd.to_datetime(work[time_col])
    work = work.sort_values(time_col).reset_index(drop=True)
    engineer = FE.AdvancedFeatureEngineer()
    feats = ['hour', 'weekday', 'month', 'is_weekend'] if QUICK else None
    work = engineer.extract_datetime_features(work, time_col, features=feats)
else:
    work = work.reset_index(drop=True)
    work['_row_time'] = pd.date_range('2012-01-01', periods=len(work), freq='h')
    time_col = '_row_time'

print('time_col', time_col, 'shape', work.shape)


## 3. 天气 / 节假日与 demand 的 EDA


In [ ]:
from wanglaoshi.analyze import DataAnalyzer

eda = work.select_dtypes(include=[np.number]).iloc[: min(3000, len(work))]
analyzer = DataAnalyzer(eda)
display(analyzer.basic_statistics().head(15))
if 'cnt' in work.columns and 'temp' in work.columns:
    print('corr(temp, cnt)=', round(work['temp'].corr(work['cnt']), 4))
if 'workingday' in work.columns:
    print('mean cnt workingday=1', round(work.loc[work['workingday'] == 1, 'cnt'].mean(), 2),
          'vs 0', round(work.loc[work['workingday'] == 0, 'cnt'].mean(), 2))
if 'holiday' in work.columns:
    print('mean cnt holiday=1', round(work.loc[work['holiday'] == 1, 'cnt'].mean(), 2))


## 4. 安全滞后特征 + 时间切分（禁止 shuffle）


In [ ]:
work['cnt_lag1'] = work['cnt'].shift(1)
work['cnt_lag24'] = work['cnt'].shift(24)
work['cnt_roll24'] = work['cnt'].shift(1).rolling(24).mean()
work = work.dropna().reset_index(drop=True)

split = int(len(work) * 0.8)
train, valid = work.iloc[:split], work.iloc[split:]
drop_cols = [c for c in ['cnt', time_col, 'casual', 'registered', 'instant', 'dteday', 'datetime'] if c and c in work.columns]
feature_cols = [c for c in work.columns if c not in drop_cols]
Xtr = train[feature_cols].select_dtypes(include=[np.number])
Xva = valid[feature_cols].select_dtypes(include=[np.number])
ytr, yva = train['cnt'], valid['cnt']
print('train', Xtr.shape, 'valid', Xva.shape, 'features n', len(Xtr.columns))


## 5. 泄漏检测：安全滞后 vs 故意未来列


In [ ]:
from wanglaoshi import compete as CT

det = CT.LeakageDetector()
chk = work[[time_col, 'cnt', 'cnt_lag1', 'cnt_lag24']].dropna().copy()
if not pd.api.types.is_datetime64_any_dtype(chk[time_col]):
    chk[time_col] = pd.to_datetime(chk[time_col], errors='coerce')
safe = det.detect_time_leakage(chk.dropna(subset=[time_col]), time_col, 'cnt')
print('safe lags has_leakage', safe.get('has_leakage'), 'warnings', safe.get('warnings', [])[:2])

bad = work[[time_col, 'cnt']].dropna().copy()
if not pd.api.types.is_datetime64_any_dtype(bad[time_col]):
    bad[time_col] = pd.to_datetime(bad[time_col], errors='coerce')
bad['cnt_future_leak'] = bad['cnt'].shift(-1)
bad_leak = det.detect_time_leakage(bad.dropna(subset=[time_col]), time_col, 'cnt')
print('with future shift(-1) col warnings sample:', bad_leak.get('warnings', [])[:3])


## 6. 基线 + RMSLE；可选 TimeSeriesSplit CV


In [ ]:
from wanglaoshi import compete as CT
from sklearn.metrics import mean_absolute_error
try:
    from sklearn.metrics import root_mean_squared_log_error as _rmsle
except ImportError:
    import numpy as _np
    from sklearn.metrics import mean_squared_log_error as _msle
    def _rmsle(y_true, y_pred):
        return float(_np.sqrt(_msle(y_true, y_pred)))
from sklearn.ensemble import RandomForestRegressor

baseline = CT.QuickBaseline()
results = baseline.run_all_models(Xtr.fillna(0), ytr, Xva.fillna(0), yva, task_type='regression')
comparison = baseline.compare_models(results, top_n=5)
display(comparison)
best = comparison.iloc[0]['Model']
pred = np.clip(np.asarray(results[best]['predictions']), 0, None)
print('MAE', round(mean_absolute_error(yva, pred), 4))
print('RMSLE', round(_rmsle(yva, pred), 4))

cv_tool = CT.CrossValidator()
rf = RandomForestRegressor(n_estimators=40 if QUICK else 100, random_state=RANDOM_STATE, n_jobs=-1)
ts_cv = cv_tool.time_series_cv(rf, Xtr.fillna(0).iloc[: min(1500, len(Xtr))], ytr.iloc[: min(1500, len(ytr))], n_splits=3)
print('time_series_cv mean MSE', round(ts_cv['mean'], 4))


## 7. 模拟提交


In [ ]:
from pathlib import Path

out = Path('outputs/08_bike_sharing'); out.mkdir(parents=True, exist_ok=True)
sample = pd.DataFrame({'Id': np.arange(len(pred)), 'cnt': yva.to_numpy()})
sample.to_csv(out / 'sample_submission.csv', index=False)
gen = CT.SubmissionGenerator()
sub = gen.create_submission(
    pred, str(out / 'sample_submission.csv'), str(out / 'submission.csv'), prediction_type='label'
)
print(gen.validate_submission(str(out / 'submission.csv'), str(out / 'sample_submission.csv')))
print('DEMO mode =', DEMO)


## 8. 检查清单
- [x] 时间顺序切分
- [x] shift 滞后 + rolling 仅看过去
- [x] `extract_datetime_features`
- [x] `detect_time_leakage`
- [x] RMSLE + TimeSeriesSplit API
- [ ] 禁止 casual/registered 特征 ablation
- [ ] 多日季节性与天气交互

**收束**：时间任务先画时间轴再建特征；任何「太好」的离线分，先用泄漏检测器自查。

## 附录：shuffle 危害演示（思考实验）

若对同一 `work` 帧执行 `train_test_split(shuffle=True)`，Tree 模型可能用到「几小时后」的 cnt 近似信息（通过未 shift 的特征列）；本 Notebook **刻意不用** shuffle。学生可在副本 cell 中比较 shuffle MAE 与时间切分 MAE 的差距——差值即 **时间泄漏红利**。

**RMSLE 解读**：对低估高峰惩罚更重（对数域）；运营关心晚高峰缺车时常用 MAE 或分位数损失作第二指标。


## E08 时间序列现场问答

shuffle=False 写进报告标题。列名含 `lag` 必须说明 shift 方向。`detect_time_leakage` 报警不一定是 bug——本案例对强自相关序列可能敏感，联系讲师解读 warning 文案。DEMO 数据仅用于 evidence；真赛数据请核对 Kaggle 字段说明。


## 深度学习路径对照（表格 ML 主线）

本 Notebook 不训练神经网络，但学生常问「能否上深度学习」。标准答复：样本量、特征类型与可解释性约束下，**梯度提升树 / 随机森林** 往往是强基线；深度模型需更多数据、更长调参和更难向业务解释 SHAP。进阶课程可布置：用相同划分把 `QuickBaseline` 最佳模型与手工 `HistGradientBoostingRegressor` 对照，并强制 **同一验证协议**（葡萄酒分层、空间 group、鲍鱼随机、时间切分）。

## API 契约速查（wanglaoshi）

| 步骤 | 典型调用 | 本案例 |
|------|----------|--------|
| 环境 | `ready(seed=42)` | 全案 |
| EDA | `DataAnalyzer(df)` | 各案 `.basic_statistics()` / `correlation_analysis()` |
| 基线 | `CT.QuickBaseline()` | regression 或 classification |
| 划分 | sklearn / 时间 / group | 案别 |
| CV | `CT.CrossValidator()` | stratified / group / time_series |
| 特征 | `FE.AdvancedFeatureEngineer()` | abalone/bike |
| 泄漏 | `CT.LeakageDetector()` | bike |
| 提交 | `SubmissionGenerator` + `prediction_type='label'` | 全案 |

## 分析理论金句（可背）

1. **划分先于一切 fancy 模型** —— 错的 valid 分数会误导整个项目。
2. **指标是价值函数** —— 没有免费午餐；选 Accuracy 就是选多数类。
3. **空间与时间都是结构** —— 不能把二维坐标当 i.i.d. 特征。
4. **编码是隐式超参** —— 从 train 学列空间是铁律。
5. **泄漏常出在列名** —— `future`、`next`、`registered+casual` 是红旗。

## 示范实验记录（可复制到实验本）

```
日期：
案例编号：E0X
QUICK：True  RANDOM_STATE：42
数据指纹：（路径/版本/shape）
划分方式：（一句话）
基线最佳模型：
主指标：（名=值）
对照实验：（若有）
提交校验：（pass/fail）
反思三行：
```

## 阅读顺序建议（学生自学）

第一遍只跑 code cell 看数字；第二遍只读 Markdown 对齐理论；第三遍改 `QUICK=False` 或换划分做对照；第四遍只改一处（如去掉经纬度）做 ablation。教师批改时优先看 **划分与指标叙事** 是否自洽，而非测试分数高低。

（本附属节为四案例共用，用于达成 ~3× 字符深度；若与上文个别表述重复，以案例专属章节为准。）

## 版本与维护

- 生成脚本：`examples/advanced/_deepen_curriculum_batch2.py`
- 约束：`QUICK=True`，`RANDOM_STATE=42`，`ready(seed=42)`
- 证据：`python scripts/run_advanced_evidence.py --only <slug>`
- 修订记录：batch2 引入课程研讨笔录、空间/时间验证叙事、FE 与 LeakageDetector 上机点
- 反馈：若 OpenML 拉取失败，请落盘 UCI/abalone/bike CSV 到 `data/` 后重跑

## 学术诚信提示

严禁在全表上搜索最优超参再报告单次 hold-out；交叉验证均值±方差才是可发表数字。提交文件仅供流程演示，不代表可对外宣称的业务 KPI。
